# Word2Vec on Dialogue Data

## Dataset

In this project, I use a dialogue dataset from the TV series *Friends*, sourced from Kaggle. The notebook keeps the original workflow and loads the dataset in the same way as the original version.


In [ ]:
from google.colab import files

uploaded = files.upload()


Saving Friends_Transcript.txt to Friends_Transcript (1).txt


## Libraries

This section imports the libraries required for preprocessing the corpus, training Word2Vec models with Gensim, and exploring the learned word embeddings.


In [ ]:
!pip install gensim

import gensim
from gensim.models import Word2Vec

import nltk
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('punkt')




[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Text Preprocessing

In this step, the text file is read and preprocessed before training the model. The preprocessing includes sentence tokenization, lowercasing, removal of stopwords, punctuation, and non-alphabetic symbols. The final result is a cleaned list of tokenized sentences, which is then used as input for Word2Vec training.


In [ ]:

stop_words = set(stopwords.words('english'))
clean_text = []

filename = "Friends_Transcript (1).txt"

with open(filename, "r", encoding="utf-8", errors="ignore") as f:
    corpus = f.read()  # read the entire file
    raw_sent = sent_tokenize(corpus)  # split it into sentences

    for sent in raw_sent:
        tokens = simple_preprocess(sent)  # split it into words

        filtered_tokens = []
        for word in tokens:
            if word not in stop_words:
                filtered_tokens.append(word)


        if filtered_tokens:
            clean_text.append(filtered_tokens)

print(len(clean_text))
print(clean_text[:5]) # print a small sample just to check that it works


115434
[['one', 'monica', 'gets', 'new', 'roomate', 'pilot', 'uncut', 'version', 'written', 'marta', 'kauffman', 'david', 'crane', 'scene', 'central', 'perk', 'chandler', 'joey', 'phoebe', 'monica'], ['monica', 'nothing', 'tell'], ['guy', 'work'], ['joey', 'mon', 'going', 'guy'], ['gotta', 'something', 'wrong']]


## Model 1: Training a CBOW Word2Vec Model

In [ ]:
# Create the model with the selected parameters
model = Word2Vec(
    vector_size=100,
    window=10,
    min_count=2,
    workers=10,
    sg=0,
    hs=0,
    negative=5
)

# Build the vocabulary
model.build_vocab(
    clean_text,
    progress_per=20000

)

print(f"Μέγεθος λεξιλογίου: {len(model.wv.index_to_key)} λέξεις")

print('Training the model...')

# train the model
model.train(
    clean_text,
    total_examples=len(clean_text),
    epochs=10,        # how many times it will see the whole dataset
    report_delay=10.0 # shows the model training progress every 10 seconds
)

print('  Done.')
print('')



Μέγεθος λεξιλογίου: 10751 λέξεις
Training the model...
  Done.



## Model 1: Exploring the Embeddings

Here, the first trained model is explored through common Word2Vec queries, such as finding similar words, measuring similarity between word pairs, identifying the word that does not match, and testing analogies.


In [ ]:
# similar words for a given word
similar_words = model.wv.most_similar('rachel', topn=10)

print("10 most similar words to 'rachel':")
print("%20s    %s" % ('word', 'score'))
for (word, score) in similar_words:
    print("%20s    %.2f" % (word, score))


# similarity between 2 words
score = model.wv.similarity('ross', 'rachel')
print("Cosine similarity between 'ross' and 'rachel' is %.2f" % score)
score = model.wv.similarity('monica', 'chandler')
print("Cosine similarity between 'monica' and 'chandler' is %.2f" % score)

# words that don't match
print(model.wv.doesnt_match(['ross', 'rachel', 'joey', 'milk']))
print(model.wv.doesnt_match(['joey', 'phoebe', 'chandler', 'hair']))

#analogy
print(model.wv.most_similar(positive=['rachel', 'monica'], negative=['joey']))



10 most similar words to 'rachel':
                word    score
            chandler    0.74
              phoebe    0.71
             richard    0.69
                hall    0.65
                kate    0.63
                ross    0.63
                mona    0.62
              monica    0.61
              deadly    0.61
                paul    0.61
Cosine similarity between 'ross' and 'rachel' is 0.63
Cosine similarity between 'monica' and 'chandler' is 0.57
milk
hair
[('phoebe', 0.6458722949028015), ('richard', 0.5373854041099548), ('barrys', 0.5183699131011963), ('phoebes', 0.5059664845466614), ('present', 0.503421425819397), ('hall', 0.5010465383529663), ('monicas', 0.5006585717201233), ('continued', 0.49938616156578064), ('hugging', 0.48984232544898987), ('deadly', 0.4882338047027588)]


## Model 2: Training and Evaluation with Different Parameters

In this section, the model is trained again using a different set of parameters in order to compare how training choices affect the learned embeddings and the type of semantic relations captured by the model.


In [ ]:
# Create the model with the selected parameters
model = Word2Vec(
    vector_size=200,
    window=5,
    min_count=3,
    workers=10,
    sg=1,
    hs=0,
    negative=5
)

# Build the vocabulary
model.build_vocab(
    clean_text,
    progress_per=20000

)

print(f"Μέγεθος λεξιλογίου: {len(model.wv.index_to_key)} λέξεις")

print('Training the model...')

# train the model
model.train(
    clean_text,
    total_examples=len(clean_text),
    epochs=10,        # how many times it will see the whole dataset
    report_delay=10.0 # shows the model training progress every 10 seconds
)

print('  Done.')
print('')

# similar words for a given word
similar_words = model.wv.most_similar('rachel', topn=10)

print("10 most similar words to 'rachel':")
print("%20s    %s" % ('word', 'score'))
for (word, score) in similar_words:
    print("%20s    %.2f" % (word, score))


# similarity between 2 words
score = model.wv.similarity('ross', 'rachel')
print("Cosine similarity between 'ross' and 'rachel' is %.2f" % score)
score = model.wv.similarity('monica', 'chandler')
print("Cosine similarity between 'monica' and 'chandler' is %.2f" % score)

# words that don't match
print(model.wv.doesnt_match(['ross', 'rachel', 'joey', 'milk']))
print(model.wv.doesnt_match(['joey', 'phoebe', 'chandler', 'hair']))

#analogy
print(model.wv.most_similar(positive=['rachel', 'monica'], negative=['joey']))



Μέγεθος λεξιλογίου: 8020 λέξεις
Training the model...
  Done.

10 most similar words to 'rachel':
                word    score
            dejected    0.73
           excitedly    0.70
                 tim    0.70
               katie    0.69
                mona    0.68
             happily    0.67
              cheryl    0.67
              relief    0.67
            panicked    0.66
                  ju    0.66
Cosine similarity between 'ross' and 'rachel' is 0.63
Cosine similarity between 'monica' and 'chandler' is 0.63
milk
hair
[('jill', 0.5173102021217346), ('dejected', 0.5155265927314758), ('tim', 0.5095280408859253), ('hugging', 0.5093875527381897), ('bless', 0.5083374381065369), ('paul', 0.5024574398994446), ('ju', 0.5001519918441772), ('excitedly', 0.4994804859161377), ('isabella', 0.49904823303222656), ('toys', 0.49886417388916016)]


## Conclusions

This project explores how Word2Vec embeddings behave when trained on a dialogue-based corpus from the TV series *Friends*.

First, the text was preprocessed through cleaning, stopword removal, lowercasing, and sentence tokenization so that it could be used as input for Word2Vec training.

Then, two Word2Vec models were trained with different parameter settings. The first model used CBOW (`sg=0`) with `vector_size=100`, `window=10`, and `min_count=2`. The second model used Skip-gram (`sg=1`) with `vector_size=200`, `window=5`, and `min_count=3`. In both cases, negative sampling was used.

The results show that the choice of parameters affects the type of semantic information that the model learns. In the first model, the nearest neighbors of words such as *rachel* were mainly other main characters, suggesting that the model captured broader narrative and co-occurrence relations across the corpus. Similarity scores such as *Ross–Rachel* and *Monica–Chandler* also support this behavior.

In the second model, the results shifted toward more local and descriptive context. Instead of mainly returning character names, the model returned words related to emotions, states, and speaking style. This suggests that the Skip-gram setting focused more on nearby contextual patterns than on broader character relationships.

Overall, the comparison between the two models shows that CBOW tends to capture more general co-occurrence patterns, while Skip-gram can capture more specific local context. This makes the project a useful example of how training choices shape the semantic structure of word embeddings.
